In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
movies =  pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv") 

## Merging Movies and credits on title

In [ ]:
movies = movies.merge(credits ,on='title')
movies=movies[['movie_id','title','overview','keywords','genres','cast','crew']]
movies.head(2)
print(movies.genres)

### The genre contains words with id . We only need the words
The below function will extract the words from the json <br>
Here ast is Abstract Syntax Tree it helps to traverse the parse tree of json  

In [ ]:
import ast
def convert(text):
    L = []
    for i in ast.literal_eval(text):
        L.append(i['name'])
    return L

        
def fetchDirector(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

In [ ]:
# Removing NaN values
movies.dropna(inplace = True)

Converting the keywords and the genre to the tags format

In [ ]:
movies['genres'] = movies['genres'].apply(convert)
movies['keywords'] = movies['keywords'].apply(convert)
movies['cast'] = movies['cast'].apply(convert)

movies.head(2)

In [ ]:
print(movies.cast[0])
print(movies.crew[0])


Same way we need top 3 cast and director from crew


In [ ]:
movies['cast'] = movies['cast'].apply(lambda x:x[0:3])
movies['crew'] = movies['crew'].apply(fetchDirector)
movies.head(2)

In [ ]:
print(movies.overview[0])

Here we remove the spaces between the names becasue if a cast or crew have same first name then it might suggest another one . So we merge their first and last name to avoid these confusion

In [ ]:
def collapse(L):
    L1=[]
    for i in L:
        L1.append(i.replace(" ",""))
    return L1

In [ ]:
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['genres'] = movies['genres'].apply(collapse)
movies['keywords'] = movies['keywords'].apply(collapse)

Spliting the string in to list 

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

Merging all to one column 'tags'

In [ ]:
movies['tags'] = movies['overview']+movies['keywords']+movies['genres']+movies['cast']+movies['crew']

In [ ]:
new_df = movies.drop(columns=['overview','keywords','genres','cast','crew'])
new_df.head(2)

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

Removing filler words

In [ ]:
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags'])

Created a matrix with 5000 most frequently used words in single movie till all the movies<br>
Eg-<br>
Movie,Action,Adventure,Love,Space,... (4996 more words)<br>
Avatar   1      1       0       1...<br>
Titanic  0       0       1       0...<br>
Iron Man 1       1       0       0,...

In [ ]:
vectors.shape

In [ ]:
similarity = cosine_similarity(vectors)
similarity = similarity.astype('float16')
similarity

In [ ]:
new_df[new_df['title'] == 'The Lego Movie'].index[0]

In [ ]:
def recommend(movie):
    index = new_df[new_df['title']== movie].index[0] # returns the indedx of the title
    distances = sorted(list(enumerate(similarity[index])),reverse=True,key=lambda x: x[1])
    for i in distances[1:6]:
        print(new_df.iloc[i[0]].title)

iloc = interger location

In [ ]:
recommend('Avatar')

The pickle library is use to serialize a obj or deserialize a obj.<br>
pickle.dump - Serialization -> convert python objbect into a byte stream<br>
picle.load - Deserialization -> converts the byte stream back to the original <br>
pickle.dump create a .pkl files only work on python <br>
##### Common Uses
- Saving machine learning models (e.g., scikit‑learn models).
- Storing preprocessed data (like similarity matrices, tokenized text).
- Caching results to avoid recomputation.
- Transferring Python objects between programs.




In [ ]:
import pickle
pickle.dump(new_df,open('movie_list.pkl','wb'))
#joblib.dump(similarity,'similarity_compressed.pkl')
pickle.dump(vectors,open('vectors.pkl','wb'))

In [ ]:
# We load the existing list to ensure we use the same data
new_df = pickle.load(open('movie_list.pkl', 'rb'))

# CRITICAL: Reset index to ensure 0, 1, 2, 3... numbering matches perfectly
new_df = new_df.reset_index(drop=True)

# Calculating Vectors
cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['tags'])

#Calculating Similarity Matrix
similarity = cosine_similarity(vectors)

#Generating Answer Key (Dictionary)
similarity_dict = {}

for i in range(len(new_df)):
    movie_title = new_df.iloc[i].title
    
    # Get Top 5 similar movies indices
    distances = sorted(list(enumerate(similarity[i])), reverse=True, key=lambda x: x[1])
    top_5_indices = [x[0] for x in distances[1:6]]
    
    # Save the result
    recommendations = []
    for index in top_5_indices:
        rec_movie = new_df.iloc[index]
        
        # Safe ID handling
        if 'movie_id' in new_df.columns:
            m_id = rec_movie.movie_id
        elif 'id' in new_df.columns:
            m_id = rec_movie.id
        else:
            m_id = rec_movie.values[0]
            
        recommendations.append((rec_movie.title, m_id))
        
    similarity_dict[movie_title] = recommendations

#Saving 'similarity_dict.pkl'
pickle.dump(similarity_dict, open('similarity_dict.pkl', 'wb'))

